In [40]:
## Create a vector of the required packages for this analysis
req_packages <- c("janitor", "tidyverse")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

## Create covariate information from metadata

In [41]:
## load in filtered snpeff results
samples <- colnames(read_tsv("input/ge.txt"))
meta <- read_csv("../analysis/input/YABLab_Drosophila_stocks.csv") %>%
    janitor::clean_names() %>%
    mutate(strain_id = str_replace_all(strain_id, "\\.", "_"))
head(meta)

Rows: 10625 Columns: 90
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (1): gene
dbl (89): BB_05_18_rep1, BB_05_18_rep2, BB_05_18_rep3, BU_06_06_rep1, BU_06_...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 294 Columns: 21
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (16): species_group, Species, strain_ID, strain_type, Genotype, label, s...
dbl  (5): year, latitude, longitude, water_distance_mi, riparian_year

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


species_group,species,strain_id,strain_type,genotype,label,year,slot,saved_on_side,locality,⋯,state,country,latitude,longitude,sequencing,wetland,water_distance_mi,water_source,riparian,riparian_year
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>
virilis,D. americana,15010-0951_00,wild-type,wt,Dame\wild-type,NA,A1-1,NA,"Anderson, Indiana.",⋯,IN,USA,NA,NA,BernardKim_nanopore,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_01,wild-type,wt,Dame\wild-type,1947,A1-2,NA,"Poplar, Montana.",⋯,MT,USA,NA,NA,NA,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_02,wild-type,wt,Dame\wild-type,1947,A1-3,NA,"Chinook, Montana.",⋯,MT,USA,NA,NA,NA,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_03,wild-type,wt,Dame\wild-type,1948,A1-4,NA,"Millersburg, Pennsylvania.",⋯,PA,USA,NA,NA,NA,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_04,wild-type,wt,Dame\wild-type,1948,A1-5,NA,"Keelers Bay, Lake Champlain, Vermont",⋯,VT,USA,NA,NA,NA,NA,NA,NA,NA,NA
virilis,D. americana,15010-0951_05,wild-type,wt,Dame\wild-type,1948,A2-1,NA,"Jackson, Michigan.",⋯,MI,USA,NA,NA,NA,NA,NA,NA,NA,NA


In [42]:
## formate metadata for covariate input for MatrixEQTL
covariate <- meta %>%
    select(strain_id, latitude, longitude, water_distance_mi, water_source) %>%
    crossing(rep = paste0("rep", 1:3)) %>%
    mutate(strain_rep = paste0(strain_id, "_", rep)) %>%
    filter(strain_rep %in% samples) %>%
    separate_wider_delim(strain_id, delim = "_", names = c("locality", "year", "female")) %>%
    select(strain_rep, locality, latitude, year, longitude, water_distance_mi, water_source) %>%
    rename(id = strain_rep) %>%
    mutate(year = as.numeric(year),
           water_source = as.numeric(factor(water_source)),
           locality = as.numeric(factor(locality))) %>%
    t() %>%
    as.data.frame() %>%
    rownames_to_column("id") %>%
    row_to_names(1)
head(covariate)

,id,BB_05_18_rep1,BB_05_18_rep2,BB_05_18_rep3,BU_06_06_rep1,BU_06_06_rep2,BU_06_06_rep3,BU_06_10_rep1,BU_06_10_rep2,BU_06_10_rep3,⋯,SV_07_02_rep3,WR_06_22_rep1,WR_06_22_rep2,WR_06_22_rep3,WR_06_44_rep1,WR_06_44_rep2,WR_06_44_rep3,WS_07_06_rep1,WS_07_06_rep2,WS_07_06_rep3
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2,locality,1,1,1,2,2,2,2,2,2,⋯,25,26,26,26,26,26,26,27,27,27
3,latitude,36.46119,36.46119,36.46119,31.99426,31.99426,31.99426,31.99426,31.99426,31.99426,⋯,39.61408,34.59891,34.59891,34.59891,34.59891,34.59891,34.59891,40.71180,40.71180,40.71180
4,year,5,5,5,6,6,6,6,6,6,⋯,7,6,6,6,6,6,6,7,7,7
5,longitude,-89.31181,-89.31181,-89.31181,-85.05089,-85.05089,-85.05089,-85.05089,-85.05089,-85.05089,⋯,-93.21157,-86.92099,-86.92099,-86.92099,-86.92099,-86.92099,-86.92099,-82.00547,-82.00547,-82.00547
6,water_distance_mi,0.04,0.04,0.04,0.00,0.00,0.00,0.00,0.00,0.00,⋯,0.01,0.03,0.03,0.03,0.03,0.03,0.03,0.62,0.62,0.62
7,water_source,3,3,3,2,2,2,2,2,2,⋯,3,2,2,2,2,2,2,3,3,3


In [43]:
## save full covariate file
write_csv(covariate, "Covariates_full.csv")

In [44]:
## subset covariate files for specific information
covariate %>%
    filter(id == "locality") %>%
    write_csv("input/Covariates_pop.csv")

covariate %>%
    filter(id == "locality" | id == "latitude") %>%
    write_csv("input/Covariates_pop_lat.csv")

## Create Covariate information for fusion status

In [45]:
## load in metadata
fusion <- read_tsv("input/strain_fusion.tsv") %>%
    janitor::clean_names()
head(fusion)

Rows: 23 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): Strain, Status

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


strain,status
<chr>,<chr>
BB.05.18,Fused
FG.06.22,Unfused
FP.10.18,Unfused
HI.99.20,Fused
II.07.04,Segregating
IR.04.34,Fused


In [46]:
## create covariate from fusion data
covariate_fusion <- fusion %>%
    mutate(strain = str_replace_all(strain, "\\.", "_")) %>%
    crossing(rep = paste0("rep", 1:3)) %>%
    mutate(strain_rep = paste0(strain, "_", rep)) %>%
    filter(strain_rep %in% samples) %>%
    separate_wider_delim(strain, delim = "_", names = c("locality", "year", "female"), cols_remove = FALSE, too_few = "align_start") %>%
    select(strain_rep, status, locality, strain) %>%
    mutate(status = as.numeric(factor(status)),
           locality = as.numeric(factor(locality)),
           strain = as.numeric(factor(strain))) %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(1) %>%
    rownames_to_column("id")
head(covariate_fusion)

,id,BB_05_18_rep1,BB_05_18_rep2,BB_05_18_rep3,FG_06_22_rep1,FG_06_22_rep2,FG_06_22_rep3,FP_10_18_rep1,FP_10_18_rep2,FP_10_18_rep3,⋯,SV_07_02_rep3,WR_06_22_rep1,WR_06_22_rep2,WR_06_22_rep3,WR_06_44_rep1,WR_06_44_rep2,WR_06_44_rep3,WS_07_06_rep1,WS_07_06_rep2,WS_07_06_rep3
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,status,1,1,1,3,3,3,3,3,3,⋯,1,3,3,3,1,1,1,1,1,1
2,locality,1,1,1,2,2,2,3,3,3,⋯,17,18,18,18,18,18,18,19,19,19
3,strain,1,1,1,2,2,2,3,3,3,⋯,18,19,19,19,20,20,20,21,21,21


In [47]:
write_csv(covariate_fusion, "input/Covariates_fusion.csv")

### Filter snp and read files for samples with fusion information

In [49]:
## load in original files
reads_original <- read_tsv("input/ge.txt")
snp_original <- read_tsv("input/snp.txt") 

head(reads_original)
head(snp_original)

Rows: 10625 Columns: 90
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (1): gene
dbl (89): BB_05_18_rep1, BB_05_18_rep2, BB_05_18_rep3, BU_06_06_rep1, BU_06_...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 3049807 Columns: 90
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (1): id
dbl (86): BB_05_18_rep1, BB_05_18_rep2, BB_05_18_rep3, BU_06_06_rep1, BU_06_...
lgl  (3): SB_02_06_rep1, SB_02_06_rep2, SB_02_06_rep3

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


gene,BB_05_18_rep1,BB_05_18_rep2,BB_05_18_rep3,BU_06_06_rep1,BU_06_06_rep2,BU_06_06_rep3,BU_06_10_rep1,BU_06_10_rep2,BU_06_10_rep3,⋯,SV_07_02_rep3,WR_06_22_rep1,WR_06_22_rep2,WR_06_22_rep3,WR_06_44_rep1,WR_06_44_rep2,WR_06_44_rep3,WS_07_06_rep1,WS_07_06_rep2,WS_07_06_rep3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
cdi,8583,8785,8738,6625,6595,6659,6446,6552,6328,⋯,7384,7539,7085,7024,7990,7339,7546,7000,7394,7385
mRpL55,214,228,256,254,222,261,285,311,336,⋯,306,276,294,267,405,427,405,354,346,381
ATPsynD,2488,2542,2837,2759,2516,2599,3015,2770,2933,⋯,3673,3335,3295,3564,3137,3587,3170,3656,3471,3797
sav,1203,1218,1218,775,741,772,839,806,971,⋯,1023,887,924,895,870,898,926,1261,1240,1205
Ctns,414,400,448,417,427,387,688,679,602,⋯,308,450,395,447,278,347,293,503,568,469
p53,265,251,266,324,311,313,393,389,397,⋯,243,362,277,310,385,336,391,325,366,306


id,BB_05_18_rep1,BB_05_18_rep2,BB_05_18_rep3,BU_06_06_rep1,BU_06_06_rep2,BU_06_06_rep3,BU_06_10_rep1,BU_06_10_rep2,BU_06_10_rep3,⋯,SV_07_02_rep3,WR_06_22_rep1,WR_06_22_rep2,WR_06_22_rep3,WR_06_44_rep1,WR_06_44_rep2,WR_06_44_rep3,WS_07_06_rep1,WS_07_06_rep2,WS_07_06_rep3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Snp_1,0,0,0,2,2,2,2,2,2,⋯,0,2,2,2,0,0,0,0,0,0
Snp_10,2,2,2,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,2,2,2
Snp_100,2,2,2,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,2,2,2
Snp_1000,0,0,0,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,0,0,0
Snp_10000,2,2,2,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,2,2,2
Snp_100000,2,2,2,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,2,2,2


In [50]:
## add replicates to dataframe and only keep columns with gene expression data
snp <- snp_original %>%
  select(id, any_of(colnames(covariate_fusion)))

reads <- reads_original %>%
    select(gene, any_of(colnames(covariate_fusion)))

head(snp)
head(reads)

id,BB_05_18_rep1,BB_05_18_rep2,BB_05_18_rep3,FG_06_22_rep1,FG_06_22_rep2,FG_06_22_rep3,FP_10_18_rep1,FP_10_18_rep2,FP_10_18_rep3,⋯,SV_07_02_rep3,WR_06_22_rep1,WR_06_22_rep2,WR_06_22_rep3,WR_06_44_rep1,WR_06_44_rep2,WR_06_44_rep3,WS_07_06_rep1,WS_07_06_rep2,WS_07_06_rep3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Snp_1,0,0,0,0,0,0,2,2,2,⋯,0,2,2,2,0,0,0,0,0,0
Snp_10,2,2,2,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,2,2,2
Snp_100,2,2,2,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,2,2,2
Snp_1000,0,0,0,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,0,0,0
Snp_10000,2,2,2,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,2,2,2
Snp_100000,2,2,2,2,2,2,2,2,2,⋯,2,2,2,2,2,2,2,2,2,2


gene,BB_05_18_rep1,BB_05_18_rep2,BB_05_18_rep3,FG_06_22_rep1,FG_06_22_rep2,FG_06_22_rep3,FP_10_18_rep1,FP_10_18_rep2,FP_10_18_rep3,⋯,SV_07_02_rep3,WR_06_22_rep1,WR_06_22_rep2,WR_06_22_rep3,WR_06_44_rep1,WR_06_44_rep2,WR_06_44_rep3,WS_07_06_rep1,WS_07_06_rep2,WS_07_06_rep3
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
cdi,8583,8785,8738,6973,7321,6982,7096,7261,7246,⋯,7384,7539,7085,7024,7990,7339,7546,7000,7394,7385
mRpL55,214,228,256,195,201,215,302,309,298,⋯,306,276,294,267,405,427,405,354,346,381
ATPsynD,2488,2542,2837,3230,3122,3125,3575,3638,3563,⋯,3673,3335,3295,3564,3137,3587,3170,3656,3471,3797
sav,1203,1218,1218,845,848,919,900,890,768,⋯,1023,887,924,895,870,898,926,1261,1240,1205
Ctns,414,400,448,363,367,369,322,271,307,⋯,308,450,395,447,278,347,293,503,568,469
p53,265,251,266,314,290,308,304,286,260,⋯,243,362,277,310,385,336,391,325,366,306


In [51]:
## save files
write_tsv(snp, "input/snp_fusion.txt")
write_tsv(reads, "input/ge_fusion.txt")